# TernaryBoost — Pipeline Completo no Colab T4
**PT-BitNet + ParetoQ/ZeroQAT + Tequila + Chat**

Tudo em ~8 minutos na GPU T4 gratuita.

In [ ]:
# 1. Setup: clone repo + install deps
!rm -rf ternary-boost
!git clone https://github.com/user/ternary-boost.git%cd ternary-boost
!pip install -q torch transformers safetensors datasets rich lm-eval wandb

import sys, os
for m in ["shared", "pt_bitnet", "paretoq", "tequila", "eval", "chat"]:
    sys.path.insert(0, f"{m}/src")

import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# 2. Escolha o modelo
MODEL_ID = "microsoft/phi-2"  # 2.7B, ~4 min na T4
# MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"  # 7B, ~12 min na T4

CACHE_DIR = "/content/cache"
LAMBDADA_GRANULARITY = "per_channel"  # ou "per_element" se quiser qualidade máxima

In [ ]:
# 3. Pipeline completo (auto-compress)
from chat.model_loader import load_model
from chat.config import ModelEntry
import time

entry = ModelEntry(
    name=MODEL_ID.split("/")[-1],
    path=MODEL_ID,
    device="cuda",
    lambada_granularity=LAMBDADA_GRANULARITY,
)

print(f"Iniciando pipeline para {MODEL_ID}...")
t0 = time.time()
model, tokenizer = load_model(entry, cache_root=CACHE_DIR)
print(f"Pipeline completo em {(time.time()-t0)/60:.1f} minutos")

# Verificar camadas
from collections import Counter
types = Counter(type(m).__name__ for m in model.modules() if "Linear" in type(m).__name__)
print(f"Camadas: {dict(types)}")

In [ ]:
# 4. Teste de qualidade com 4 prompts
prompts = [
    "What is the capital of France?",
    "Explain machine learning in one sentence.",
    "Write a haiku about the moon.",
    "If a train goes 60mph for 2 hours, how far does it travel?",
]

model.eval()
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(inputs.input_ids, max_new_tokens=30, temperature=0.7,
                            do_sample=True, pad_token_id=tokenizer.eos_token_id)
    elapsed = time.time() - t0
    gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    tok_count = out.shape[1] - inputs.input_ids.shape[1]
    print(f"\nQ: {prompt}")
    print(f"A: {gen.strip()[:150]}")
    print(f"   ({tok_count} tokens, {elapsed:.1f}s, {tok_count/elapsed:.1f} tok/s)")

In [ ]:
# 5. Chat interativo simples
print("CHAT MODE (digite 'quit' para sair)")
print("="*50)

conversation = []
while True:
    user = input("\nYou: ")
    if user.lower() == 'quit':
        break
    
    conversation.append(f"User: {user}")
    prompt_text = "\n".join(conversation) + "\nAssistant:"
    
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        out = model.generate(inputs.input_ids, max_new_tokens=80, temperature=0.7,
                            do_sample=True, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"Assistant: {response.strip()}")
    conversation.append(f"Assistant: {response.strip()}")